# Experimental Models - Rent Price Prediction

In [1]:

from dotenv import load_dotenv
from mlModels.regression.data.data import getData, getRegressionData
from mlModels.kmeans.runCluster import runCluster
import os

CV = True
CV_VAL = 5

os.chdir("/home/florian/Desktop/immopreis-regression")

load_dotenv("database/.env")

df_features = getData(filter_type="finance_type", filter_val="rent", table="rent_features")
cluster_dict = runCluster()

/home/florian/Desktop/immopreis-regression/mlModels/regression/data/data.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_features = pd.read_sql("SELECT * FROM rent_features", conn)
/home/florian/Desktop/immopreis-regression/mlModels/regression/data/data.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_listings = pd.read_sql("SELECT * FROM listings", conn)
/home/florian/Desktop/immopreis-regression/mlModels/kmeans/data/data.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM lis


-------Starting K-Means Clustering locations-------


# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getLinearRegressionData(df)

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    model = LinearRegression()

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------Linear Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    print()

# Lasso

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getLinearRegressionData(df)

    model = Lasso(alpha=0.1)

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("-------Lasso Regression Results-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")



# Ridge

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")

    df_house_X, df_house_y, df_apt_X, df_apt_y = getRegressionData(df)
    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    model = Ridge(alpha=0.1)

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------Ridge Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    print()

# Random Forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")

    df_house_X, df_house_y, df_apt_X, df_apt_y = getRegressionData(df)
    df_apt_X = df_apt_X.drop(columns=["is_mfh", "is_efh", "is_lh", "is_villa", "is_dhh",
                                      "is_sbc", "is_rh", "is_ab", "is_bh", "is_gh"])
    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    drop_cols = ["is_wg", "is_phw", "is_ceiling", "is_electro", "has_kitchen",
                 "estate_ratio", "log_wintergarden_size", "is_bio", "fgee",
                 "wintergarden_ratio", "loc_14_6", "has_wintergarden", "loc_14_7",
                 "is_geothermal", "loc_14_8", "is_photovoltaik", "is_oven", "is_gw", "is_gc",
                 "loc_14_4", "loc_14_13", "is_urban", "loc_14_5", "loc_14_10", "loc_14_2",
                 "is_infrared", "is_central", "loc_14_11", "loc_14_0", "is_ms", "is_pellets",
                 "is_oil", "is_air_heating", "loc_14_3", "loc_14_1", "is_dgw", "log_garden_size",
                 "has_garden", "log_loggia_size", "has_loggia", "loc_14_12", "is_apt", "is_egw",
                 "has_balcony", "garden_ratio", "has_carport", "has_closet", "has_cellar"]
    X_train.drop(columns=drop_cols, inplace=True, errors="ignore")
    X_test.drop(columns=drop_cols, inplace=True, errors="ignore")

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------Random Forest Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    importance_df = pd.DataFrame({
        "feature" : X_train.columns,
        "importance" : model.feature_importances_
    })
    importance_df = importance_df.sort_values("importance", ascending=False)
    print(importance_df.tail(20))
    print()


# XGBoost

In [7]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from xgboost import XGBRegressor as xgbr

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getRegressionData(df)

    drop_cols = ["is_mfh", "is_efh", "is_lh", "is_villa", "is_dhh",
                 "is_sbc", "is_rh", "is_ab", "is_bh", "is_gh"]

    df_apt_X.drop(columns=drop_cols,  inplace=True, errors="ignore")
    df_apt_y.drop(columns=drop_cols, inplace=True, errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    drop_cols = ["finance_type", "is_wg", "log_wintergarden_size", "fgee", "is_phw", "wintergarden_ratio", "estate_ratio", "is_geothermal", "is_ceiling", "is_bio", "is_electro",
                 "has_kitchen", "is_infrared", "is_oil", "is_gw", "is_air_heating", "is_gc", "is_central", "loc_4_0", "garden_ratio", "has_loggia", "is_dgw", "has_garden",
                 "loc_14_7", "loc_14_6", "loc_14_8"]

    X_train.drop(columns=drop_cols, inplace=True, errors="ignore")
    X_test.drop(columns=drop_cols, inplace=True, errors="ignore")

    model = xgbr(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------XGBoost Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    importance_df = pd.DataFrame({
        "feature" : X_train.columns,
        "importance" : model.feature_importances_
    })
    importance_df = importance_df.sort_values("importance", ascending=False)
    print(importance_df.tail(20))
    print()


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.9s finished



-------Cross Validation Scores for K=3-------
CV R2 Scores: [0.58111556 0.63025485 0.65573583 0.71974537 0.57459105]
Mean R2 Score: 0.6323
Standard Deviation: 0.0532

-------XGBoost Regression Results for K=3-------
MAE:  0.1278
RMSE: 0.1589
R2:   0.7646
                      feature  importance
24           log_terrace_size    0.024954
1                 has_carport    0.024669
26  log_distance_nearest_city    0.021415
30                    loc_3_0    0.020493
17                     is_egw    0.017833
14              balcony_ratio    0.016838
0                       rooms    0.016760
29    log_distance_klagenfurt    0.015436
16                   is_urban    0.012759
13                    is_oven    0.012515
7                 has_balcony    0.011784
5                 has_parking    0.010613
19                     is_apt    0.010598
23            log_garden_size    0.010344
4                  has_cellar    0.009843
10                 is_pellets    0.009737
25            log_loggia_size 

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.9s finished



-------Cross Validation Scores for K=4-------
CV R2 Scores: [0.58206327 0.64448918 0.66817676 0.70482111 0.59125912]
Mean R2 Score: 0.6382
Standard Deviation: 0.0463

-------XGBoost Regression Results for K=4-------
MAE:  0.1298
RMSE: 0.1615
R2:   0.7569
                    feature  importance
1               has_carport    0.030684
29  log_distance_klagenfurt    0.028030
14            balcony_ratio    0.027690
0                     rooms    0.024186
31                  loc_4_2    0.023740
11          is_photovoltaik    0.020911
17                   is_egw    0.017980
7               has_balcony    0.016351
4                has_cellar    0.014878
19                   is_apt    0.014791
25          log_loggia_size    0.014549
16                 is_urban    0.014189
5               has_parking    0.013248
10               is_pellets    0.013157
23          log_garden_size    0.013115
18                    is_ms    0.011821
6                has_closet    0.010961
13                  is_o

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.1s finished
